In [1]:
!pip install requests pymongo tqdm

In [2]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")

print(client.list_database_names())

['admin', 'config', 'local']


In [3]:
db = client["sansad"]
collection = db["attendance"]

collection.insert_one({
    "name": "Kausik",
    "attendance": "Present",
    "date": "2024-07-01"
})

InsertOneResult(ObjectId('69e45c747b210cdf0efe6377'), acknowledged=True)

In [4]:
import requests
import time
from pymongo import MongoClient

client = MongoClient("mongodb://127.0.0.1:27017/")
db = client["sansad"]
collection = db["attendance"]

BASE_URL = "https://sansad.in/api_ls/member"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Referer": "https://sansad.in/"
}

def fetch_attendance(loksabha, session, date):
    url = f"{BASE_URL}/getMemberAttendanceDateWise"
    
    params = {
        "loksabha": loksabha,
        "session": session,
        "dateOfAttendance": date,
        "locale": "en"
    }

    res = requests.get(url, headers=HEADERS, params=params)
    
    if res.status_code == 200:
        return res.json()
    return None

# Save to MongoDB

In [5]:
def save_to_mongo(data, loksabha, session, date):
    if not data:
        return
    
    members = data.get("members", [])
    
    for m in members:
        doc = {
            "name": m.get("name"),
            "member_id": m.get("memberId"),
            "attendance": m.get("attendance"),
            "date": date,
            "session": session,
            "loksabha": loksabha
        }

        collection.update_one(
            {
                "member_id": doc["member_id"],
                "date": doc["date"]
            },
            {"$set": doc},
            upsert=True
        )

In [7]:
# Running for one test date
data = fetch_attendance(17, 1, "2024/07/02")

save_to_mongo(data, 17, 1, "2024/07/02")

print("Done")

Done


In [8]:
for doc in collection.find().limit(5):
    print(doc)

{'_id': ObjectId('69e45c747b210cdf0efe6377'), 'name': 'Kausik', 'attendance': 'Present', 'date': '2024-07-01'}


In [9]:
data = fetch_attendance(17, 1, "2024/07/02")

print(data)

[]


In [10]:
test_dates = [
    "2024/07/01",
    "2024/07/22",
    "2024/07/23",
    "2024/08/01"
]

for d in test_dates:
    data = fetch_attendance(17, 1, d)
    print(d, "->", len(data))

2024/07/01 -> 0
2024/07/22 -> 0
2024/07/23 -> 0
2024/08/01 -> 0


In [11]:
for session in range(10, 20):
    data = fetch_attendance(17, session, "2024/07/22")
    print("Session", session, "->", len(data))

Session 10 -> 0
Session 11 -> 0
Session 12 -> 0
Session 13 -> 0
Session 14 -> 0
Session 15 -> 0
Session 16 -> 0
Session 17 -> 0
Session 18 -> 0
Session 19 -> 0


In [14]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64)",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://sansad.in/",
    "Origin": "https://sansad.in"
}

In [15]:
def fetch_memberwise(loksabha, session):
    url = "https://sansad.in/api_ls/member/getMemberAttendanceMemberWise"
    
    params = {
        "loksabha": loksabha,
        "session": session,
        "locale": "en"
    }

    res = requests.get(url, headers=HEADERS, params=params)

    print("Status:", res.status_code)  # debug

    if res.status_code == 200:
        return res.json()
    else:
        print(res.text)  # 🔥 important to see error
        return None

In [16]:
data = fetch_memberwise(18, 5)

if data:
    print("Length:", len(data))
    print(data[:2])

Status: 200
Length: 544
[{'mpsno': 4589, 'memberName': 'Narendra Modi', 'constituency': 'Varanasi', 'state': 'Uttar Pradesh', 'stateCode': 'UP', 'signedDaysCount': 0, 'division': '1'}, {'mpsno': 4268, 'memberName': 'Rajnath Singh', 'constituency': 'Lucknow', 'state': 'Uttar Pradesh', 'stateCode': 'UP', 'signedDaysCount': 0, 'division': '2'}]


# Storing this in MongoDB

In [17]:
def save_memberwise(data, loksabha, session):
    for m in data:
        doc = {
            "mp_id": m.get("mpsno"),
            "name": m.get("memberName"),
            "constituency": m.get("constituency"),
            "state": m.get("state"),
            "attendance_days": m.get("signedDaysCount"),
            "loksabha": loksabha,
            "session": session
        }

        collection.update_one(
            {
                "mp_id": doc["mp_id"],
                "session": session
            },
            {"$set": doc},
            upsert=True
        )

In [18]:
data = fetch_memberwise(18, 5)

save_memberwise(data, 18, 5)

print("Saved to MongoDB 🚀")

Status: 200
Saved to MongoDB 🚀


In [19]:
for doc in collection.find().limit(5):
    print(doc)

{'_id': ObjectId('69e45c747b210cdf0efe6377'), 'name': 'Kausik', 'attendance': 'Present', 'date': '2024-07-01'}
{'_id': ObjectId('69e45ee1243043f738d8f730'), 'mp_id': 4589, 'session': 5, 'attendance_days': 0, 'constituency': 'Varanasi', 'loksabha': 18, 'name': 'Narendra Modi', 'state': 'Uttar Pradesh'}
{'_id': ObjectId('69e45ee1243043f738d8f732'), 'mp_id': 4268, 'session': 5, 'attendance_days': 0, 'constituency': 'Lucknow', 'loksabha': 18, 'name': 'Rajnath Singh', 'state': 'Uttar Pradesh'}
{'_id': ObjectId('69e45ee1243043f738d8f734'), 'mp_id': 5021, 'session': 5, 'attendance_days': 0, 'constituency': 'Gandhinagar', 'loksabha': 18, 'name': 'Amit Shah', 'state': 'Gujarat'}
{'_id': ObjectId('69e45ee1243043f738d8f736'), 'mp_id': 4923, 'session': 5, 'attendance_days': 0, 'constituency': 'Nagpur', 'loksabha': 18, 'name': 'Nitin Jairam Gadkari', 'state': 'Maharashtra'}


In [20]:
collection.delete_one({"name": "Kausik"})

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)

# Top Attendance MPs

In [21]:
top_mps = collection.find().sort("attendance_days", -1).limit(5)

for mp in top_mps:
    print(mp["name"], mp["attendance_days"])

Ganesh Singh 21
Nishikant Dubey 21
Ramvir Singh Bidhuri 21
Biplab Kumar Deb 21
Trivendra Singh Rawat 21


# State-wise analysis

In [22]:
pipeline = [
    {
        "$group": {
            "_id": "$state",
            "total_attendance": {"$sum": "$attendance_days"}
        }
    },
    {"$sort": {"total_attendance": -1}}
]

for doc in collection.aggregate(pipeline):
    print(doc)

{'_id': 'Uttar Pradesh', 'total_attendance': 1414}
{'_id': 'Maharashtra', 'total_attendance': 806}
{'_id': 'Tamil Nadu', 'total_attendance': 706}
{'_id': 'Bihar', 'total_attendance': 617}
{'_id': 'West Bengal', 'total_attendance': 575}
{'_id': 'Madhya Pradesh', 'total_attendance': 476}
{'_id': 'Gujarat', 'total_attendance': 448}
{'_id': 'Rajasthan', 'total_attendance': 401}
{'_id': 'Karnataka', 'total_attendance': 400}
{'_id': 'Andhra Pradesh', 'total_attendance': 400}
{'_id': 'Odisha', 'total_attendance': 370}
{'_id': 'Kerala', 'total_attendance': 366}
{'_id': 'Telangana', 'total_attendance': 287}
{'_id': 'Assam', 'total_attendance': 250}
{'_id': 'Jharkhand', 'total_attendance': 220}
{'_id': 'Punjab', 'total_attendance': 219}
{'_id': 'Chhattisgarh', 'total_attendance': 201}
{'_id': 'Haryana', 'total_attendance': 142}
{'_id': 'NCT of Delhi', 'total_attendance': 124}
{'_id': 'Himachal Pradesh', 'total_attendance': 83}
{'_id': 'Uttarakhand', 'total_attendance': 77}
{'_id': 'Jammu And Kas

# inactive MPs

In [23]:
inactive = collection.find({"attendance_days": 0})

for mp in inactive.limit(5):
    print(mp["name"])

Narendra Modi
Rajnath Singh
Amit Shah
Nitin Jairam Gadkari
Pralhad Joshi


In [24]:
!pip install streamlit pymongo pandas

  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cachetools-7.0.5-py3-none-any.whl.metadata (5.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-manylinux2014_x86_64.whl.metadata (44 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.3-py3-none-any.whl.metadata (4.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 1.5 MB/s eta 0:00:00m eta 0:00:010:00:010m
Using cached altair-6.0.0-py3-none-any.whl (795 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached cachetools-7.0.5-py3-none-any.whl (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 2

In [25]:
def fetch_members(loksabha=18):
    url = "https://sansad.in/api_ls/member"

    params = {
        "loksabha": loksabha,
        "page": 1,
        "size": 600,  # get all MPs
        "locale": "en"
    }

    res = session_req.get(url, params=params)

    if res.status_code == 200:
        return res.json()
    else:
        print(res.text)
        return None

In [26]:
def save_members(data, loksabha):
    for m in data:
        doc = {
            "mp_id": m.get("mpsno"),
            "name": m.get("memberName"),
            "party": m.get("party"),
            "state": m.get("state"),
            "constituency": m.get("constituency"),
            "loksabha": loksabha
        }

        collection.update_one(
            {"mp_id": doc["mp_id"]},
            {"$set": doc},
            upsert=True
        )